# 01 — Case study 1 — germinating spores: Cellpose 3

Trains a **Cellpose 3** model (`cyto2`) on `case_study_1/mycol_saved_session_CS1.zip` at mycol's default settings, runs the default
Optuna hyperparameter search, and scores the result on the held-out test images.

**Split:** 652 images, 521 train / 131 test, 3,415 test cells (min_cells = 1)

Both models are trained and tuned by **mycol's own worker scripts** (`unified_worker.py`), driven
through the same npz interface the app uses — not a re-implementation. Cellpose 3's workers live on
the `main` branch and Cellpose 4's on `cp4`, so each runs from its own checkout with its own
environment:

| | Cellpose 3 | Cellpose 4 |
|---|---|---|
| checkout | `../mycol-main-cp3` (worktree of `main`) | this repo (`cp4`) |
| cellpose | 3.1.1 | 4.2.1.1 |
| base model | `cyto2` | `cpsam_v2` |
| epochs | 100 | 100 |
| learning rate / weight decay | 0.1 / 1e-4 | 1e-5 / 0.1 |
| batch size | 8 | 1 |
| images per epoch | all training images | 8 |
| Optuna trials | 20 | 20 |

Every value above is mycol's own default for that generation, taken from each branch's
`fine_tune_panel.py`. Each branch's Optuna search space is used unchanged — including Cellpose 4's
`diameter` search, which Cellpose 3 does not have.

**The train/test split is the saved session's**, rebuilt from `image_metadata.json` key order (mycol's
`ordered_keys()`, which is upload order and not always alphabetical) with the session's own
`min_cells_per_image`. That parameter decides which images are eligible, so it has to come from the
session for the split to match; everything else uses the defaults above.

Because Cellpose 3 and 4 cannot share an interpreter, the whole pipeline runs in a subprocess using
the matching environment. **This notebook therefore runs under any Python kernel** — it needs no
Cellpose itself.

### Recorded

Training seconds, tuning seconds, the chosen hyperparameters, and on the test set: **R²**, **MAE**
and **MAPE** on per-image cell counts, **mean IoU** of matched objects, **AP@0.5/0.75/0.9**, **F1@0.5**
and inference seconds per image. Results are written to `results/cs1_cp3.json` and reused on re-run
(pass `force=True` to recompute).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import common

DATASET, MODEL = "cs1", "cp3"
session = common.load_session(DATASET)
print(f"dataset        : {DATASET}")
print(f"image size     : {session['shape']}")
print(f"min_cells      : {session['min_cells']}  (from the saved session, so the split matches)")
print(f"eligible images: {len(session['kept'])} of {len(session['images'])}")
print(f"  train        : {len(session['train']):4}  ({sum(session['counts'][n] for n in session['train']):,} annotated cells)")
print(f"  test         : {len(session['test']):4}  ({sum(session['counts'][n] for n in session['test']):,} annotated cells)")
print()
print("mycol defaults for", MODEL, "->", common.DEFAULTS[MODEL])

## Train, tune, evaluate

This runs mycol's finetune worker, then its validation worker (Optuna), then scores the tuned model on
the test split. Expect this to take a while — Cellpose 4's tuning in particular is dominated by
inference cost.

In [ ]:
record = common.run_case(DATASET, MODEL)

In [ ]:
# --- what was measured ---
import json

keys = ["n_train", "n_test", "train_cells", "test_cells", "train_seconds", "tune_seconds",
        "n_trials_run", "infer_seconds_per_image", "r2", "mae", "mape",
        "mean_iou_matched", "ap50", "ap75", "ap90", "f1_50"]
for k in keys:
    v = record[k]
    print(f"  {k:26} {v:.4f}" if isinstance(v, float) else f"  {k:26} {v}")
print(f"\n  best hyperparameters      {json.dumps(record['best_params'])}")

In [ ]:
# --- true vs predicted cell count on the held-out test images ---
import matplotlib.pyplot as plt
import numpy as np

t = np.array(record["true_counts"]); p = np.array(record["pred_counts"])
fig, ax = plt.subplots(figsize=(4.8, 4.8))
lim = max(t.max(), p.max()) * 1.05
ax.plot([0, lim], [0, lim], ls="--", lw=1.2, color="#8a8a8a")
ax.scatter(t, p, s=38, alpha=0.75, color={"cp3": '"#0072B2"', "cp4": '"#D55E00"'}["cp3"],
           edgecolors="white", linewidths=0.5)
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal")
ax.set_xlabel("true count"); ax.set_ylabel("predicted count")
ax.set_title("cs1 — Cellpose 3", fontsize=11, fontweight="bold")
ax.text(0.05, 0.95, f"R² {record['r2']:.3f}\nMAE {record['mae']:.1f}\nMAPE {record['mape']:.1f}%",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dddddd"))
ax.grid(True, lw=0.4, color="#ededed"); ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout(); plt.show()